# 02 · Light, unidades responsáveis e dados do COR

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore/blob/main/02_light_e_cor.ipynb)

**Projeto:** Chamados de poda de árvore no Rio de Janeiro &nbsp;·&nbsp; **Parte 2 de 5**

---

### Contexto
Parte dos problemas causados por árvores envolve a rede elétrica: queda sobre fiação, poste e falta de luz. A pergunta é se os dados abertos permitem ligar um chamado de poda à atuação da **Light**.

### Pergunta
> É possível identificar, só com dados do Data.Rio, quando uma poda envolve a rede elétrica?

### Dados utilizados
- `adm_central_atendimento_1746.chamado`
- `adm_cor_comando.ocorrencias`
- `adm_cor_comando.ocorrencias_orgaos_responsaveis`
- `adm_cor_comando.procedimento_operacional_padrao`

### Roteiro
1. Quem atende os chamados de poda no 1746
2. Unidades ligadas a energia e justificativas
3. Tabelas do Centro de Operações (COR)
4. Integridade das tabelas e eventos de árvore / rede elétrica

### Principais resultados
- A poda é atendida pela **COMLURB**; a Light não aparece como unidade no 1746.
- Nenhuma justificativa de chamado de poda menciona Light, fiação ou rede elétrica.
- No COR, a Light tem **5.407 acionamentos em 2.940 eventos**, sobretudo em queda de árvore (1.103) e de poste (345).
- A tabela do COR **termina em 02/04/2024**, reaproveita `id_evento` entre anos e tem 1.060 datas zeradas.

> ℹ️ As saídas deste notebook são as da execução original no Google Colab (dados extraídos do BigQuery
> em novembro de 2026), então os resultados podem ser lidos sem rodar nada. Para reproduzir, rode a
> célula de configuração abaixo (veja o [README](README.md)).

In [ ]:
# Configuração (rode primeiro)
import os
import subprocess
import sys
from pathlib import Path

# No Colab, informe o projeto do Google Cloud que vai pagar a leitura (veja o README):
# os.environ["GCP_PROJECT_ID"] = "seu-projeto-gcp"

RAIZ = Path.cwd()
if not (RAIZ / "src" / "podas").exists():  # aberto direto no Colab: baixa o código do repositório
    RAIZ = Path("/content/analise-podas")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore.git", str(RAIZ)], check=True)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from google.cloud import bigquery

from podas.bq import get_client, rodar
from podas.config import ALVO, SENSIBILIDADE, ARQ_PODA, DATA_DIR, FIG_DIR, OUT_DIR

client = get_client()

## 1. Quem atende os chamados de poda no 1746

In [ ]:
IDS = ["790", "118"]  # candidatos investigados (o resultado mostra que 118 é a COMLURB)

# A) o que as unidades 790 e 118 atendem, em qualquer subtipo
sql_a = """
SELECT
  id_unidade_organizacional AS id_un,
  nome_unidade_organizacional AS nome_un,
  id_unidade_organizacional_mae AS id_mae,
  tipo, subtipo,
  MIN(EXTRACT(YEAR FROM data_inicio)) AS ano_min,
  MAX(EXTRACT(YEAR FROM data_inicio)) AS ano_max,
  COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE id_unidade_organizacional IN UNNEST(@ids)
   OR id_unidade_organizacional_mae IN UNNEST(@ids)
GROUP BY 1, 2, 3, 4, 5
ORDER BY n DESC
LIMIT 60
"""

# B) quem atende os subtipos de poda (e os dois candidatos ligados à rede)
PODA_REDE = ALVO + SENSIBILIDADE + [
    "Remoção de galho sobre a rede ou luminária",
    "Poda de árvore por obstrução da Iluminação Pública",
]
sql_b = """
SELECT
  id_unidade_organizacional AS id_un,
  nome_unidade_organizacional AS nome_un,
  subtipo,
  COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@poda)
GROUP BY 1, 2, 3
ORDER BY n DESC
LIMIT 60
"""

un = rodar(sql_a, [bigquery.ArrayQueryParameter("ids", "STRING", IDS)])
print("=== O QUE 790 E 118 ATENDEM ===")
print(un.to_string())

poda_un = rodar(sql_b, [bigquery.ArrayQueryParameter("poda", "STRING", PODA_REDE)])
print("\n=== QUEM ATENDE OS SUBTIPOS DE PODA ===")
print(poda_un.to_string())

custo estimado: 2.51 GB
=== O QUE 790 E 118 ATENDEM ===
   id_un                                          nome_un id_mae                                                           tipo                                                                                          subtipo  ano_min  ano_max        n
0    118  COMLURB - Companhia Municipal de Limpeza Urbana     26                                               Remoção Gratuita                                                            Remoção de entulho e bens inservíveis     2011     2025  2536073
1    118  COMLURB - Companhia Municipal de Limpeza Urbana     26                               Controle de roedores e caramujos                                                       Controle de roedores e caramujos africanos     2011     2026   386168
2    118  COMLURB - Companhia Municipal de Limpeza Urbana     26                                         Limpeza de logradouros                                                             

- A unidade **118 é a COMLURB**, e não a Rio Luz; a 790 não aparece.
- A poda é atendida quase só pela **COMLURB (118)** e pela **Ouvidoria COMLURB (247)**.
- Os subtipos ligados à rede ("Remoção de galho sobre a rede ou luminária", "Poda de árvore por
  obstrução da Iluminação Pública") são raríssimos (35 e 3 chamados) e atendidos pela **RIOLUZ (8)**.

## 2. Unidades ligadas a energia e justificativas dos chamados de poda

In [ ]:
import re

# 1) catálogo completo de unidades (sem filtro por subtipo) e busca por nome
sql = """
SELECT id_unidade_organizacional AS id_un,
       nome_unidade_organizacional AS nome_un,
       id_unidade_organizacional_mae AS id_mae,
       COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
GROUP BY 1, 2, 3
"""
cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
print(f"custo estimado: {client.query(sql, job_config=cfg).total_bytes_processed/1e9:.2f} GB")
cfg = bigquery.QueryJobConfig(maximum_bytes_billed=20 * 10**9)
unidades_all = client.query(sql, job_config=cfg).to_dataframe()
print(f"unidades distintas: {len(unidades_all)}; soma: {unidades_all['n'].sum():,}")

pad = re.compile(r"light|energia|el[eé]tric|concession|rede|fia[cç][aã]o", re.I)
achou = unidades_all[unidades_all["nome_un"].fillna("").str.contains(pad)]
print("\n=== UNIDADES COM NOME LIGADO A LIGHT/ENERGIA ===")
print(achou.sort_values("n", ascending=False).to_string())

# 2) texto de justificativa nos chamados de poda
sql2 = """
SELECT subtipo, status, situacao, SUBSTR(justificativa_status, 1, 200) AS just, COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@alvo)
  AND REGEXP_CONTAINS(LOWER(IFNULL(justificativa_status,'')),
      r'light|fia[cç][aã]o|fios?\b|rede el[eé]trica|rede aérea|energia|concession')
GROUP BY 1, 2, 3, 4
ORDER BY n DESC
LIMIT 40
"""
cfg = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("alvo", "STRING", ALVO + SENSIBILIDADE)],
    maximum_bytes_billed=20 * 10**9,
)
just = client.query(sql2, job_config=cfg).to_dataframe()
print("\n=== JUSTIFICATIVAS COM MENÇÃO A LIGHT/FIAÇÃO ===")
print(just.to_string())

# 3) tabelas do COR (para o caso de a Light aparecer como órgão responsável)
print("\n=== TABELAS DO adm_cor_comando ===")
for t in client.list_tables("datario.adm_cor_comando"):
    print(t.table_id)

custo estimado: 0.97 GB
unidades distintas: 868; soma: 20,263,796

=== UNIDADES COM NOME LIGADO A LIGHT/ENERGIA ===
    id_un                                               nome_un id_mae        n
428     8  RIOLUZ - Companhia Municipal de Energia e Iluminação     26  1582879
83   1693         TR/SUBP/CGR - Coordenadoria de Gestão de Rede   1686      134

=== JUSTIFICATIVAS COM MENÇÃO A LIGHT/FIAÇÃO ===
Empty DataFrame
Columns: [subtipo, status, situacao, just, n]
Index: []

=== TABELAS DO adm_cor_comando ===
ocorrencias
ocorrencias_orgaos_responsaveis
procedimento_operacional_padrao
procedimento_operacional_padrao_orgaos_responsaveis


Nenhuma unidade do 1746 corresponde à Light, e **nenhuma justificativa** de chamado de poda menciona
Light, fiação ou rede elétrica. O 1746 sozinho não permite identificar podas que dependem da Light.

## 3. Tabelas do Centro de Operações (COR)

In [ ]:
for nome in ["ocorrencias", "ocorrencias_orgaos_responsaveis"]:
    t = client.get_table(f"datario.adm_cor_comando.{nome}")
    print(f"\n=== {nome} ({t.num_rows:,} linhas) ===")
    for c in t.schema:
        print(" ", c.name, c.field_type)

for nome in ["ocorrencias", "ocorrencias_orgaos_responsaveis"]:
    print(f"\n=== AMOSTRA: {nome} ===")
    print(client.list_rows(f"datario.adm_cor_comando.{nome}", max_results=10)
          .to_dataframe().to_string())


=== ocorrencias (91,406 linhas) ===
  id_pop STRING
  id_evento STRING
  bairro STRING
  data_inicio TIMESTAMP
  data_fim TIMESTAMP
  prazo STRING
  descricao STRING
  gravidade STRING
  latitude FLOAT
  longitude FLOAT
  status STRING
  tipo STRING
  data_particao DATE

=== ocorrencias_orgaos_responsaveis (255,570 linhas) ===
  id_evento STRING
  sigla STRING
  data_chegada TIMESTAMP
  data_inicio TIMESTAMP
  data_fim TIMESTAMP
  descricao STRING
  status STRING
  data_particao DATE
  primary_key STRING

=== AMOSTRA: ocorrencias ===
  id_pop id_evento bairro               data_inicio                  data_fim prazo                                                       descricao gravidade   latitude  longitude   status      tipo data_particao
0      1     27219   None 2018-06-01 16:57:59+00:00 2018-06-01 18:09:27+00:00  None  Av. nossa sra. de copacabana, alt. r. sá ferreira - copacabana     Baixo -22.971110 -43.186846  Fechado  Primario    2018-06-01
1      1     27221   None 2018-06

In [ ]:
import re

# 1) todas as siglas de órgãos acionados (procura a Light aqui)
sql1 = """
SELECT sigla,
       COUNT(*) AS n,
       COUNT(DISTINCT id_evento) AS eventos,
       MIN(EXTRACT(YEAR FROM data_inicio)) AS ano_min,
       MAX(EXTRACT(YEAR FROM data_inicio)) AS ano_max
FROM `datario.adm_cor_comando.ocorrencias_orgaos_responsaveis`
GROUP BY sigla
ORDER BY n DESC
"""
siglas = rodar(sql1)
print("=== SIGLAS ===")
print(siglas.to_string())

# 2) ações (descricao) que mencionam árvore, rede, energia, Light
sql2 = """
SELECT sigla, descricao, COUNT(*) AS n
FROM `datario.adm_cor_comando.ocorrencias_orgaos_responsaveis`
WHERE REGEXP_CONTAINS(LOWER(IFNULL(descricao,'')),
      r'[aá]rvore|galho|poda|rede|fia[cç][aã]o|fio|energia|light|el[eé]tric|poste')
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 40
"""
print("\n=== AÇÕES COM MENÇÃO A ÁRVORE/REDE/ENERGIA ===")
print(rodar(sql2).to_string())

# 3) cobertura de ocorrencias por ano: bairro preenchido e coordenada válida
sql3 = """
SELECT EXTRACT(YEAR FROM data_inicio) AS ano,
       COUNT(*) AS n,
       COUNTIF(bairro IS NOT NULL AND TRIM(bairro) != '') AS bairro_ok,
       COUNTIF(latitude BETWEEN -23.08 AND -22.74
               AND longitude BETWEEN -43.80 AND -43.09) AS coord_ok
FROM `datario.adm_cor_comando.ocorrencias`
GROUP BY ano ORDER BY ano
"""
print("\n=== COBERTURA DE ocorrencias POR ANO ===")
print(rodar(sql3).to_string())

# 4) esquema e amostra dos procedimentos (dão o nome do id_pop)
for nome in ["procedimento_operacional_padrao", "procedimento_operacional_padrao_orgaos_responsaveis"]:
    t = client.get_table(f"datario.adm_cor_comando.{nome}")
    print(f"\n=== {nome} ({t.num_rows:,} linhas) ===")
    print([(c.name, c.field_type) for c in t.schema])
    print(client.list_rows(f"datario.adm_cor_comando.{nome}", max_results=60)
          .to_dataframe().to_string())

custo estimado: 0.006 GB
=== SIGLAS ===
                          sigla       n  eventos  ano_min  ano_max
0                       Cet-rio  127597    65712     1970     2024
1                     Bombeiros   43676    24108     1970     2023
2                            Pm   25271    23833     1970     2024
3                       Comlurb   14509    10089     1970     2024
4                            Gm   12413     7980     1970     2024
5                         Light    5407     2940     1970     2024
6                    Seconserva    5372     5047     1970     2024
7                         Lamsa    4435     2289     1970     2024
8                     Portonovo    3431     1919     1970     2023
9                Concessionária    3297     1915     1970     2024
10                Policia civil    3046     2912     1970     2023
11                       Rioluz    1680      920     1970     2024
12                        Cedae    1439      920     1970     2023
13                  De

No COR a Light aparece como órgão acionado: **5.407 acionamentos em 2.940 eventos**. As ações mais
comuns são desligar energia, restabelecer fornecimento, retirar poste e verificar fiação.
O campo `bairro` das ocorrências fica vazio até 2018.

## 4. Integridade das tabelas e eventos de árvore / rede elétrica

In [ ]:
OC  = "`datario.adm_cor_comando.ocorrencias`"
ORG = "`datario.adm_cor_comando.ocorrencias_orgaos_responsaveis`"
POP = "`datario.adm_cor_comando.procedimento_operacional_padrao`"

# 1) integridade da chave id_evento e datas zeradas
sql1 = f"""
SELECT
  (SELECT COUNT(*) FROM {OC}) AS n_ocorrencias,
  (SELECT COUNT(DISTINCT id_evento) FROM {OC}) AS n_eventos_distintos,
  (SELECT COUNT(DISTINCT id_evento) FROM {ORG}) AS n_eventos_em_orgaos,
  (SELECT COUNT(DISTINCT id_evento) FROM {ORG}
     WHERE id_evento IN (SELECT id_evento FROM {OC})) AS n_orgaos_que_casam,
  (SELECT COUNTIF(data_inicio < TIMESTAMP '2000-01-01') FROM {ORG}) AS orgaos_data_zerada,
  (SELECT COUNTIF(data_inicio < TIMESTAMP '2000-01-01') FROM {OC}) AS oc_data_zerada
"""
print("=== INTEGRIDADE ===")
print(rodar(sql1).T.to_string())

# 2) eventos de árvore/rede/apagão por ano, com Light e Concessionária acionadas
sql2 = f"""
WITH acion AS (
  SELECT id_evento,
         LOGICAL_OR(LOWER(TRIM(sigla)) = 'light') AS light,
         LOGICAL_OR(LOWER(TRIM(sigla)) IN ('concessionária','concessionaria')) AS concess
  FROM {ORG}
  GROUP BY id_evento
)
SELECT o.id_pop, p.pop_titulo,
       EXTRACT(YEAR FROM o.data_inicio) AS ano,
       COUNT(DISTINCT o.id_evento) AS eventos,
       COUNT(DISTINCT IF(a.light, o.id_evento, NULL)) AS com_light,
       COUNT(DISTINCT IF(a.concess, o.id_evento, NULL)) AS com_concess
FROM {OC} o
LEFT JOIN acion a USING (id_evento)
LEFT JOIN {POP} p USING (id_pop)
WHERE o.id_pop IN ('11', '12', '17', '40')
  AND o.data_inicio >= TIMESTAMP '2015-01-01'
GROUP BY 1, 2, 3
ORDER BY id_pop, ano
"""
print("\n=== EVENTOS DE ÁRVORE / POSTE / APAGÃO POR ANO ===")
print(rodar(sql2).to_string())

# 3) em que tipos de evento a Light é acionada
sql3 = f"""
SELECT o.id_pop, p.pop_titulo, COUNT(DISTINCT o.id_evento) AS eventos
FROM {ORG} g
JOIN {OC} o USING (id_evento)
LEFT JOIN {POP} p USING (id_pop)
WHERE LOWER(TRIM(g.sigla)) = 'light'
GROUP BY 1, 2
ORDER BY eventos DESC
"""
print("\n=== TIPOS DE EVENTO EM QUE A LIGHT É ACIONADA ===")
print(rodar(sql3).to_string())

# 4) volume mensal recente (2024 é parcial ou a tabela parou?)
sql4 = f"""
SELECT FORMAT_TIMESTAMP('%Y-%m', data_inicio) AS mes, COUNT(*) AS n
FROM {OC}
WHERE data_inicio >= TIMESTAMP '2023-07-01'
GROUP BY mes ORDER BY mes
"""
print("\n=== OCORRÊNCIAS POR MÊS (2023-07 em diante) ===")
print(rodar(sql4).to_string())

# 5) nomes de bairro do COR nos eventos de árvore, e esquema da tabela mestre de bairros
sql5 = f"""
SELECT bairro, COUNT(*) AS n
FROM {OC}
WHERE id_pop IN ('11', '40') AND data_inicio >= TIMESTAMP '2021-01-01'
GROUP BY bairro ORDER BY n DESC LIMIT 25
"""
print("\n=== BAIRROS (COR) NOS EVENTOS DE ÁRVORE, 2021+ ===")
print(rodar(sql5).to_string())

tb = client.get_table("datario.dados_mestres.bairro")
print(f"\n=== dados_mestres.bairro ({tb.num_rows:,} linhas) ===")
print([(c.name, c.field_type) for c in tb.schema])
print(client.list_rows("datario.dados_mestres.bairro", max_results=5).to_dataframe().drop(columns=["geometry"], errors="ignore").to_string())

=== INTEGRIDADE ===
custo estimado: 0.005 GB
                         0
n_ocorrencias        91406
n_eventos_distintos  87620
n_eventos_em_orgaos  84780
n_orgaos_que_casam   84551
orgaos_data_zerada    1060
oc_data_zerada           0

=== EVENTOS DE ÁRVORE / POSTE / APAGÃO POR ANO ===
custo estimado: 0.006 GB
   id_pop                    pop_titulo   ano  eventos  com_light  com_concess
0      11               Queda de árvore  2015      184         98            0
1      11               Queda de árvore  2016      313        130            1
2      11               Queda de árvore  2017      268        110            2
3      11               Queda de árvore  2018      385        117            0
4      11               Queda de árvore  2019      865        170            3
5      11               Queda de árvore  2020      441        165            3
6      11               Queda de árvore  2021      599        188            5
7      11               Queda de árvore  2022      413   

- `id_evento` não é chave única em `ocorrencias` (91.406 linhas para 87.620 eventos).
- 1.060 acionamentos têm data zerada (1970).
- A Light é acionada sobretudo em **queda de árvore** (1.103 eventos) e **queda de poste** (345).
- A participação da Light nas quedas de árvore despenca em 2023–2024 (11 e 5 eventos), e o
  procedimento "Queda de árvore sobre fiação" só existe a partir de 2021: mudanças de registro,
  não necessariamente do fenômeno.

In [ ]:
OC = "`datario.adm_cor_comando.ocorrencias`"

# 1) por que id_evento se repete em ocorrencias
sql1 = f"""
SELECT id_evento, COUNT(*) AS n
FROM {OC}
GROUP BY id_evento
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 5
"""
dup_ids = rodar(sql1)
print("=== EVENTOS DUPLICADOS ===")
print(dup_ids.to_string())

if len(dup_ids):
    sql2 = f"""
    SELECT * FROM {OC}
    WHERE id_evento = '{dup_ids.iloc[0]['id_evento']}'
    """
    print(rodar(sql2).to_string())

# 2) a tabela realmente para em abril de 2024, ou é lacuna?
sql3 = f"""
SELECT MAX(data_inicio) AS ultima_data, MAX(data_particao) AS ultima_particao
FROM {OC}
"""
print("\n=== ÚLTIMA DATA NA TABELA ===")
print(rodar(sql3).to_string())

custo estimado: 0.001 GB
=== EVENTOS DUPLICADOS ===
  id_evento  n
0      6309  2
1      6372  2
2      6350  2
3      6381  2
4      6342  2
custo estimado: 0.012 GB
  id_pop id_evento       bairro               data_inicio                  data_fim    prazo                                   descricao gravidade   latitude  longitude   status      tipo data_particao
0      1      6309         None 2016-06-01 10:38:40+00:00 2016-06-01 10:49:33+00:00     None             Av. das américas, 5001 - barra      Baixo -23.008476 -43.444183  Fechado  Primario    2016-06-01
1      1      6309  Cosme velho 2024-02-22 09:19:57+00:00                       NaT  83548.0  Tunel reboucas, 1deg galeria - cosme velho     Média -22.958531 -43.204386  Fechado  Primario    2024-02-22

=== ÚLTIMA DATA NA TABELA ===
custo estimado: 0.001 GB
                ultima_data ultima_particao
0 2024-04-02 23:37:51+00:00      2024-04-02


O `id_evento` é reaproveitado entre anos (o mesmo ID aparece em 2016 e em 2024), e **a tabela termina
em 2 de abril de 2024**.

**Conclusão:** os dados abertos do COR cobrem cerca de 5 mil acionamentos da Light, com lacunas de
registro e sem continuidade após abril de 2024. Para responder à pergunta sobre a Light é preciso
obter dados da própria concessionária ou do COR atualizados.

---

⬅️ [01 · Catálogo de subtipos e definição do recorte de poda](01_catalogo_de_subtipos.ipynb) &nbsp;|&nbsp; [🏠 Início](README.md) &nbsp;|&nbsp; [03 · Qualidade dos dados e extração base](03_qualidade_e_extracao.ipynb) ➡️